# FG-TRACER: fgtracer-medgemma

Traces **MedGemma-4B** only. Loads the model from **HuggingFace**
(`google/medgemma-4b-it`, gated) using the Kaggle Secret `HF_TOKEN` (the same one
the v7 run used). Work is split across both T4s and merged. One model per
session keeps you far under the 12h cap.

**Setup:** attach the `fgtracer-code` dataset; enable Internet; have the
`HF_TOKEN` secret set (Add-ons -> Secrets) with MedGemma license accepted on HF.

In [ ]:
# 1. Deps. Single clean pip pass (no destructive uninstall, no --no-deps numpy
#    hack -- those left transformers half-installed so GenerationMixin failed to
#    import). Pin transformers to the 4.56 line (requirements: >=4.50,<5) and let
#    pip resolve the rest in ONE pass. Fail loudly on the exact imports we need.
%pip install -q -U "transformers>=4.56,<4.57" "huggingface_hub>=0.34" accelerate "datasets>=2.19" pillow safetensors
import transformers, torch
from transformers import AutoModelForImageTextToText
from transformers.generation import GenerationMixin
print('deps OK | transformers', transformers.__version__,
      '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())


In [ ]:
# extract/copy FG-TRACER code into the working dir (handles zip OR loose files)
import glob, os, shutil, zipfile
WORK='/kaggle/working'
zips=glob.glob('/kaggle/input/**/fgtracer_code.zip',recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(WORK)
else:
    src=glob.glob('/kaggle/input/**/run_tracing.py',recursive=True)
    assert src,'attach the fgtracer-code dataset'
    cd=os.path.dirname(src[0])
    for it in os.listdir(cd):
        s=os.path.join(cd,it); d=os.path.join(WORK,it)
        if os.path.isdir(s): shutil.copytree(s,d,dirs_exist_ok=True)
        elif not os.path.exists(d): shutil.copy2(s,d)
os.chdir(WORK)
import fgtracer  # fail loudly if the package did not land
print('code ready:',sorted(os.listdir(WORK))[:12])

In [ ]:
# HF token loaded from private Kaggle dataset (not hardcoded in source).
# W&B key still from Kaggle Secrets (it's only used for logging, not model download).
import os, pathlib
HF_TOKEN = pathlib.Path('/kaggle/input/fgtracer-hf-token/hf_token.txt').read_text().strip()
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
# W&B: try Kaggle secret; if unavailable, skip logging gracefully
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret('WANDB_API_KEY')
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print('HF token loaded | W&B authenticated')
except Exception as _e:
    print(f'HF token loaded | W&B skipped ({_e})')

CKPT = 'google/medgemma-4b-it'
print('will load from HF:', CKPT)


In [ ]:
# Trace, split across both T4s (disjoint ranges) -> two dirs -> merge
import subprocess, os, torch
OUT='outputs_medgemma'
common=['--checkpoint_dir',CKPT,'--device','cuda','--dtype','bfloat16','--quantization','none',
        '--block_types','image_to_last,question_to_last','--k','1','--max_new_tokens','64']
jobs=[('0',['--start_index','0','--num_samples','50','--answer_path',OUT+'_g0']),
      ('1',['--start_index','50','--num_samples','50','--answer_path',OUT+'_g1'])]
procs=[]
for gpu,extra in jobs:
    if int(gpu)>=torch.cuda.device_count(): continue
    env=dict(os.environ,CUDA_VISIBLE_DEVICES=gpu)
    procs.append(subprocess.Popen(['python','run_tracing.py']+common+extra,env=env))
for p in procs: p.wait()
parts=[d for d in (OUT+'_g0',OUT+'_g1') if os.path.isdir(d)]
subprocess.run(['python','merge_outputs.py','--inputs',*parts,'--output',OUT],check=True)
print('merged ->',OUT,sorted(os.listdir(OUT)))

In [ ]:
import json, glob, os
for f in sorted(glob.glob('outputs_medgemma/results_*_k1.json')):
    d=json.load(open(f)); rs=d['results']
    print(os.path.basename(f),'| n=',len(rs),'| mnt=',d['meta']['max_new_tokens'],
          '| acc=',round(d['run_summary'].get('accuracy') or 0,3))
    print('   gen:',repr(rs[0]['generated'][:70]))

In [ ]:
# W&B cloud logging is optional: a missing or rejected key must not fail the run
import json, os, glob
try:
    import wandb
    run = wandb.init(project="fgtracer", name="fgtracer-medgemma-kaggle",
                     tags=["medgemma", "tracing", "kaggle"],
                     config={"model": "google/medgemma-4b-it", "n_samples": 100, "k": 1, "split": "vqa_rad"})
    for f in sorted(glob.glob("outputs_medgemma/results_*_k1.json")):
        d = json.load(open(f)); rs = d.get("results", [])
        block = "image_to_last" if "image" in f else "question_to_last"
        wandb.log({f"n_samples/{block}": len(rs),
                   f"accuracy/{block}": d.get("run_summary", {}).get("accuracy", 0)})
    artifact = wandb.Artifact("outputs-medgemma", type="results", description="MedGemma-4B tracing outputs")
    artifact.add_dir("outputs_medgemma")
    run.log_artifact(artifact)
    wandb.finish()
    print("W&B run logged:", run.url)
except Exception as _e:
    print(f'W&B logging skipped ({_e})')
